# 13 · Formula refinement, round 2

Starting point: `SPEC3c` from `12_formula` (40 parameters, 15-fold AUC 0.7195); best submission `blend_v3` (0.7 formula + 0.3 CatBoost, 0.7204).

| Step | Question |
|---|---|
| A | Do any columns dropped early (`02`) carry an **era-specific** effect the pooled tests missed? |
| B | Is 2017-18 one regime, and is the boundary exactly 2017-01-01? |
| C | Are the rule definitions exact (age band edges and which age, education rule, drivers-per-vehicle cuts, credit step)? |
| D | Which of the 40 parameters are redundant? |
| E | Is anything left, including 3+-way interactions? (deep residual GBMs, targeted 3-way terms) |
| F | Confirm, blend, submit |

Protocol as in `12`: decisions on the seed-0 repeat (paired Δ AUC and Δ log-loss ×10⁴, positive = better), confirmation on seeds 1–2.

In [1]:
import json, sys, itertools, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
sys.path.insert(0, '.')
import common as C, features as FE, formula as FM
import formula_steps as FS
from formula_steps import t, run, run4, cmp, table_fixed, replace, rescope, drop, nparams, SPEC3c, SPEC4
pd.set_option('display.width', 200)
L, y, lab, hold = FS.L, FS.y, FS.lab, FS.hold
PC = json.load(open('params_cats.json'))
cb15 = C.cv_score('cats15___copies__ohm_10', FE.add_cats(FE.add_static(lab)), PC['features'], 'cb', PC['params'])
B = SPEC3c
b0 = run(B, 0.1)
fmt = {'auc': '{:.5f}', 'Δauc': '{:+.5f}', 'se_auc': '{:.5f}', 'Δll(+better)': '{:+.2f}', 'se_ll': '{:.2f}', 'z_ll': '{:+.1f}', 'params': '{:.0f}'}
def tab(rows):
    tb = pd.DataFrame(rows).T
    tb['z_ll'] = tb['Δll(+better)'] / tb['se_ll'].replace(0, np.nan)
    return tb
print(f'SPEC3c: seed-0 AUC {b0["auc_mean"]:.4f}, {nparams(B)} parameters')

SPEC3c: seed-0 AUC 0.7194, 40 parameters


## A. Never-tried columns, each offered for all eras / early vs late / per era (seed 0, C = 0.1)

In [2]:
CAND = ['QUOTE_CHANNEL', 'MULTI_QUOTE_IND', 'DAYS_TO_FOLLOWUP', 'PRIMARY_PARKING', 'PRIOR_CLAIMS_CNT', 'TOTAL_NUM_VEH', 'NUM_OWNED_VEH',
        'NUM_LOANED_VEH', 'NUM_LEASED_VEH', 'DRV_MALE_SHARE', 'DRV_MARRIED_SHARE', 'DRV_TELEMATICS_ANY', 'DRV_TELEMATICS_SHARE',
        'DRV_DEPENDENT_SHARE', 'DRV_SENIOR_SHARE', 'VEH_AGE_MIN', 'VEH_AGE_MEAN', 'VEH_AGE_MAX', 'VEH_MILEAGE_SUM', 'VEH_MILEAGE_MAX',
        'VEH_SAFETY_MIN', 'VEH_SAFETY_MEAN', 'VEH_GARAGE_SHARE', 'VEH_ANTI_THEFT_SHARE', 'VEH_N_MAKES', 'ACS_MED_HH_INCOME',
        'ACS_EDU_NO_HS', 'ACS_EDU_BACHELOR_PLUS', 'ACS_MED_AGE', 'ACS_COMMUTE_WFH', 'QUOTE_MONTH', 'QUOTE_DOW', 'STATE_CD',
        'HOME_POLICY_IND', 'VEH_N_ROWS', 'DRV_HIGH_EDU_SHARE', 'DRV_OWN_SHARE', 'DRV_RENT_SHARE', 'DRV_SAFETY_MEAN', 'DRV_ACC_FREE_MEAN',
        'DRV_COMMUTE_SUM', 'DRV_YOUNG_SHARE', 'ACS_OWNER_OCC_SHARE', 'ACS_VEH_PER_HH', 'ACS_COMMUTE_TRANSIT', 'ACS_COMMUTE_MEAN_MIN']
CATS = {'QUOTE_CHANNEL', 'PRIMARY_PARKING', 'STATE_CD', 'QUOTE_DOW'}
V = {f'{c} @{sc}': B + [t(c, 'onehot', sc) if c in CATS else t(c, 'linear', sc, na=bool(L[c].isna().any()))]
     for c in CAND for sc in ('all', 'split', 'per-era')}
ta = table_fixed(V, B, 0.1); ta['z_ll'] = ta['Δll(+better)'] / ta['se_ll'].replace(0, np.nan)
print(f'{len(V)} tests; largest log-loss z = {ta.z_ll.max():.2f} (≈ what the max of {len(V)} pure-noise tests looks like)')
ta.sort_values('Δll(+better)', ascending=False).head(12).style.format(fmt)

138 tests; largest log-loss z = 1.37 (≈ what the max of 138 pure-noise tests looks like)


,auc,Δauc,se_auc,Δll(+better),se_ll,params,z_ll
VEH_AGE_MIN @all,0.71950,+0.00008,0.00025,+0.86,0.63,42,+1.4
VEH_AGE_MIN @split,0.71953,+0.00010,0.00026,+0.76,0.75,43,+1.0
DAYS_TO_FOLLOWUP @split,0.71994,+0.00051,0.00031,+0.60,1.03,42,+0.6
VEH_AGE_MIN @per-era,0.71944,+0.00002,0.00030,+0.52,0.78,44,+0.7
VEH_AGE_MEAN @all,0.71941,-0.00002,0.00038,+0.50,0.84,42,+0.6
DAYS_TO_FOLLOWUP @per-era,0.71988,+0.00045,0.00037,+0.30,1.20,43,+0.3
QUOTE_CHANNEL @per-era,0.71958,+0.00015,0.00032,+0.30,0.86,49,+0.4
VEH_AGE_MEAN @split,0.71933,-0.00010,0.00035,+0.28,0.77,43,+0.4
NUM_OWNED_VEH @all,0.71962,+0.00019,0.00009,+0.14,0.38,41,+0.4
DRV_SENIOR_SHARE @split,0.71951,+0.00008,0.00021,+0.10,0.58,42,+0.2


**Reading:** **nothing hidden.** None of the 46 columns adds signal in any era scope: the largest log-loss gain is 1.4 se across 138 tests, which is what the best of 138 pure-noise tests looks like. Channel, multi-quote, follow-up days, prior claims, gender and marital mix, telematics, vehicle age/mileage/safety, Census income and education, month and weekday all stay out, as do state and the terms dropped in `12`.

## B. Era boundaries

In [3]:
def by_year(spec, col, form):
    out = []
    for x in spec:
        if (x['col'], x['form']) == (col, form) and x['scope'] == 'late':
            out += [dict(x, scope='Y2017'), dict(x, scope='Y2018')]
        elif (x['col'], x['form']) == (col, form) and x['scope'] in ('per-era', 'split'):
            out.append(dict(x, scope='per-year'))
        else:
            out.append(x)
    return out
r4 = lambda sp, ic: C.cv_score('fx', L, FM.columns(sp), 'formula', {'spec': sp, 'C': 0.1, 'intercepts': ic}, seeds=(0,), log=False)
rows = {'intercepts: 4 years': cmp(r4(B, '4era'), b0)}
terms = [(x['col'], x['form']) for x in B if x['scope'] in ('per-era', 'split', 'late')]
for col, form in terms:
    rows[f'4 intercepts + {col} by year'] = cmp(r4(by_year(B, col, form), '4era'), b0)
dt = pd.to_datetime(L.QUOTE_DT)
for cut in ['2016-07-01', '2016-10-01', '2017-04-01', '2017-07-01']:
    L2 = L.copy(); L2['ERA'] = np.where(dt >= cut, '2017-18', np.where(L2.QUOTE_YEAR <= 2015, '2015', '2016'))
    rows[f'late era starts {cut} (instead of 2017-01-01)'] = cmp(C.cv_score(f'fx_cut{cut}', L2, FM.columns(B), 'formula', {'spec': B, 'C': 0.1}, seeds=(0,), log=False), b0)
tab(rows).style.format(fmt)

,auc,Δauc,se_auc,Δll(+better),se_ll,z_ll
intercepts: 4 years,0.71943,+0.00001,0.00046,-0.47,1.38,-0.3
4 intercepts + QUOTED_AMT by year,0.71990,+0.00047,0.00067,+0.46,2.02,+0.2
4 intercepts + ACS_POP_DENSITY by year,0.71944,+0.00001,0.00051,-0.52,1.55,-0.3
4 intercepts + R_EDU_MAJ by year,0.71937,-0.00005,0.00045,-0.65,1.33,-0.5
4 intercepts + NUM_DRIVERS by year,0.71906,-0.00037,0.00058,-1.42,1.66,-0.9
4 intercepts + R_DPV_GT1 by year,0.71987,+0.00045,0.00052,+0.52,1.64,+0.3
4 intercepts + R_DPV_HALF_TO_1 by year,0.71954,+0.00011,0.00042,-0.35,1.27,-0.3
4 intercepts + VEH_LUXURY_SHARE by year,0.71957,+0.00015,0.00046,-0.45,1.32,-0.3
late era starts 2016-07-01 (instead of 2017-01-01),0.70623,-0.01320,0.00229,-53.44,6.02,-8.9
late era starts 2016-10-01 (instead of 2017-01-01),0.71079,-0.00863,0.00195,-39.22,5.35,-7.3


**Reading:**
- **2017 and 2018 are one regime:** separate intercepts or per-year terms change nothing (all within 1 se).
- **The boundary is exactly 2017-01-01:** moving it by a quarter in either direction costs 0.003–0.013 AUC (log-loss z −7 to −9).

## C. Exact rule definitions (each replaces the current rule; seed 0)

In [4]:
dr = pd.read_parquet(C.PROC / 'drivers.parquet')
g = dr.assign(a=dr.DRIVER_AGE.between(35, 50)).groupby('POLICY_ID')
L['N_AGE_35_50'] = L.POLICY_ID.map(g.a.sum())
alt_age = {'any driver 35-50': (L.N_AGE_35_50 > 0), 'all drivers 35-50': (L.N_AGE_35_50 == L.NUM_DRIVERS),
           'median age 35-50': L.POLICY_ID.map(g.DRIVER_AGE.median()).between(35, 50),
           'youngest 35-50': L.DRV_AGE_MIN.between(35, 50), 'oldest 35-50': L.DRV_AGE_MAX.between(35, 50)}
for a in (33, 34, 35, 36, 37):
    for z in (48, 49, 50, 51, 52):
        if (a, z) != (35, 50): alt_age[f'mean age in [{a},{z}]'] = L.DRV_AGE_MEAN.between(a, z)
alt_age['mean age in [35,50)'] = (L.DRV_AGE_MEAN >= 35) & (L.DRV_AGE_MEAN < 50)
ne, nd = L.N_EDU_DRIVERS, L.NUM_DRIVERS
alt_edu = {'share >= 0.5': L.DRV_HIGH_EDU_SHARE >= 0.5, '>= 1 educated': ne >= 1, '>= 2 educated': ne >= 2, 'all educated': ne == nd}
dpv = L.DRIVERS_PER_VEH
alt_dpv = {f'band [{lo},{hi}] + >{hi}': (dpv.between(lo, hi), dpv > hi) for lo in (0.33, 0.5, 0.67) for hi in (1.0, 1.25, 1.5) if (lo, hi) != (0.5, 1.0)}
alt_dpv['band [0.5,1) + >=1'] = ((dpv >= 0.5) & (dpv < 1), dpv >= 1)
rows = {}
for n, v in alt_age.items():
    col = 'ALT_' + ''.join(ch if ch.isalnum() else '_' for ch in n); L[col] = v.astype(int)
    rows[f'age: {n}'] = cmp(run(replace(B, 'DRV_AGE_35_50', [t(col, 'flag')]), 0.1), b0)
for n, v in alt_edu.items():
    col = 'ALT_EDU_' + ''.join(ch if ch.isalnum() else '_' for ch in n); L[col] = v.astype(int)
    rows[f'education (late): {n}'] = cmp(run(replace(B, 'R_EDU_MAJ', [t(col, 'flag', 'late')]), 0.1), b0)
for i, (n, (band, gt)) in enumerate(alt_dpv.items()):
    c1, c2 = f'ALT_DPVB_{i}', f'ALT_DPVG_{i}'
    L[c1], L[c2] = band.astype(int), gt.astype(int)
    rows[f'drivers/vehicle: {n}'] = cmp(run([x for x in B if x['col'] not in ('R_DPV_GT1', 'R_DPV_HALF_TO_1')] + [t(c1, 'flag', 'per-era'), t(c2, 'flag', 'per-era')], 0.1), b0)
nostep = [x for x in B if not (x['col'] == 'CREDIT_SCORE' and x['form'] == 'step')]
for cut in (550, 575, 600, 625, 675, 700, 725, 750):
    rows[f'credit step at {cut} (2015)'] = cmp(run(nostep + [t('CREDIT_SCORE', 'step', '2015', cuts=[cut])], 0.1), b0)
for sc in ('early', 'per-era', 'all'):
    rows[f'credit step 650 @{sc}'] = cmp(run(nostep + [t('CREDIT_SCORE', 'step', sc, cuts=[650])], 0.1), b0)
for k in (3000, 5000, 7500, 10000):
    rows[f'price: + step at {k} per era'] = cmp(run(B + [t('QUOTED_AMT', 'step', 'per-era', cuts=[k])], 0.1), b0)
for k in (5000, 10000, 20000, 30000):
    rows[f'density: + hinge at {k} per era'] = cmp(run(B + [t('ACS_POP_DENSITY', 'hinge', 'per-era', knots=[k])], 0.1), b0)
tc = tab(rows)
display(tc.sort_values('Δll(+better)', ascending=False).head(8).style.format(fmt))
print('worst alternatives (current rule is better by this much):')
tc.sort_values('Δll(+better)').head(8).style.format(fmt)

,auc,Δauc,se_auc,Δll(+better),se_ll,z_ll
price: + step at 7500 per era,0.71973,+0.00030,0.00041,+0.61,0.96,+0.6
price: + step at 5000 per era,0.71943,+0.00000,0.00014,+0.09,0.38,+0.2
density: + hinge at 10000 per era,0.71962,+0.00020,0.00021,+0.08,0.51,+0.2
density: + hinge at 30000 per era,0.71953,+0.00010,0.00009,-0.14,0.37,-0.4
density: + hinge at 5000 per era,0.71948,+0.00006,0.00011,-0.17,0.20,-0.9
credit step 650 @per-era,0.71933,-0.00009,0.00024,-0.28,0.38,-0.8
density: + hinge at 20000 per era,0.71949,+0.00007,0.00012,-0.28,0.39,-0.7
price: + step at 3000 per era,0.71944,+0.00001,0.00011,-0.48,0.27,-1.8


worst alternatives (current rule is better by this much):


,auc,Δauc,se_auc,Δll(+better),se_ll,z_ll
age: oldest 35-50,0.70477,-0.01466,0.00198,-43.06,7.86,-5.5
"drivers/vehicle: band [0.67,1.5] + >1.5",0.70717,-0.01225,0.00094,-38.20,3.52,-10.8
age: any driver 35-50,0.70685,-0.01258,0.00183,-36.84,7.36,-5.0
age: youngest 35-50,0.70788,-0.01155,0.00178,-35.30,6.31,-5.6
education (late): >= 1 educated,0.70937,-0.01006,0.00152,-34.50,3.46,-10.0
education (late): share >= 0.5,0.70984,-0.00959,0.00157,-33.10,4.01,-8.3
"drivers/vehicle: band [0.33,1.5] + >1.5",0.70898,-0.01045,0.00116,-32.41,3.07,-10.6
education (late): >= 2 educated,0.70984,-0.00959,0.00219,-32.40,5.43,-6.0


**Reading: every current rule is the sharp optimum.**
- **Age band:** mean driver age in [35, 50] inclusive beats every other edge (34–52) and every other age definition (any, all, youngest, oldest, median: −0.002 to −0.015).
- **Education:** strictly "more than half" beats ≥ half, ≥ 1 or ≥ 2 educated, and all educated (−0.003 to −0.010).
- **Drivers per vehicle:** the cuts at exactly 0.5 and 1 are best.
- **Credit:** a step at exactly 650, in 2015 only, is best.
- **Price and density:** no extra step or hinge helps.

These look like literal constants in the generator.

## D. Parsimony (seed 0)

In [5]:
noNA = [dict(x, na=False) if x.get('na') else x for x in B]
V = {'pool 2015+2016 intercepts': (B, '2era'), 'drop missing-value flags': (noNA, '3era'),
     'drivers/vehicle bands: 2015 = 2016': (rescope(rescope(B, 'R_DPV_GT1', 'flag', 'split'), 'R_DPV_HALF_TO_1', 'flag', 'split'), '3era'),
     'luxury share: 2017-18 only': (rescope(B, 'VEH_LUXURY_SHARE', 'linear', 'late'), '3era'),
     'package as one ordinal term': (replace(B, 'COV_PKG_TYPE', [t('PKG_ORDINAL', 'linear')]), '3era'),
     'drop credit slope (keep 2015 step)': ([x for x in B if not (x['col'] == 'CREDIT_SCORE' and x['form'] == 'linear')], '3era'),
     'price: 2015 = 2016': (rescope(B, 'QUOTED_AMT', 'linear', 'split'), '3era'),
     'drop drivers (late)': ([x for x in B if x['col'] != 'NUM_DRIVERS'] + [t('NUM_DRIVERS', 'linear', 'early')], '3era')}
rows = {}
for n, (sp, ic) in V.items():
    rows[n] = cmp(r4(sp, ic), b0) | {'params': FM.Formula(sp, 1, ic).fit(L.iloc[:4000], y[:4000]).n_params}
display(tab(rows).style.format(fmt))
print('SPEC4 = SPEC3c with all changes above except "price: 2015 = 2016" and "drop drivers (late)":')
for x in SPEC4: print('  ', x)
print('intercepts:', FS.SPEC4_INTERCEPTS, '| parameters:', FM.Formula(SPEC4, 1, '2era').fit(L.iloc[:4000], y[:4000]).n_params)

,auc,Δauc,se_auc,Δll(+better),se_ll,params,z_ll
pool 2015+2016 intercepts,0.71949,+0.00006,0.00008,+0.04,0.24,39,+0.2
drop missing-value flags,0.71996,+0.00053,0.00024,+1.24,0.70,36,+1.8
drivers/vehicle bands: 2015 = 2016,0.71957,+0.00015,0.00007,+0.39,0.22,38,+1.8
luxury share: 2017-18 only,0.71934,-0.00009,0.00012,-0.07,0.50,39,-0.1
package as one ordinal term,0.71980,+0.00037,0.00021,+1.21,0.69,38,+1.7
drop credit slope (keep 2015 step),0.71991,+0.00048,0.00032,+0.64,0.83,38,+0.8
price: 2015 = 2016,0.71916,-0.00026,0.00018,-0.39,0.45,39,-0.9
drop drivers (late),0.71936,-0.00007,0.00014,-0.53,0.31,39,-1.7


SPEC4 = SPEC3c with all changes above except "price: 2015 = 2016" and "drop drivers (late)":
   {'col': 'CREDIT_SCORE', 'form': 'step', 'scope': '2015', 'cuts': [650]}
   {'col': 'QUOTED_AMT', 'form': 'linear', 'scope': 'per-era', 'na': False}
   {'col': 'ACS_POP_DENSITY', 'form': 'linear', 'scope': 'per-era', 'na': False}
   {'col': 'R_EDU_MAJ', 'form': 'flag', 'scope': 'late'}
   {'col': 'DISCOUNT_IND', 'form': 'flag', 'scope': '2016'}
   {'col': 'NUM_DRIVERS', 'form': 'linear', 'scope': 'split'}
   {'col': 'DRV_AGE_35_50', 'form': 'flag', 'scope': 'all'}
   {'col': 'R_DPV_GT1', 'form': 'flag', 'scope': 'split'}
   {'col': 'R_DPV_HALF_TO_1', 'form': 'flag', 'scope': 'split'}
   {'col': 'PRIOR_CARRIER_GRP', 'form': 'onehot', 'scope': 'all'}
   {'col': 'CAT_ZONE', 'form': 'linear', 'scope': 'all', 'na': False}
   {'col': 'VEH_LUXURY_SHARE', 'form': 'linear', 'scope': 'late'}
   {'col': 'PKG_ORDINAL', 'form': 'linear', 'scope': 'all'}
intercepts: 2era | parameters: 29


Carrier levels could be merged into 5–6 groups (≈ +0.0004 on seed 0), but the groups come from coefficients fitted on all labeled data, so that gain is optimistic. Not adopted.

In [6]:
wide = {Cr: run4(SPEC4, Cr) for Cr in (0.01, 0.03, 0.1, 0.3, 1, 10)}
display(pd.DataFrame({Cr: {'seed-0 AUC': r['auc_mean'], 'seed-0 log-loss': r['fold_ll'].mean()} for Cr, r in wide.items()}).T.style.format('{:.5f}'))
s3, s4 = run(SPEC3c, 0.1, seeds=(0, 1, 2)), run4(SPEC4, 0.1, seeds=(0, 1, 2))
rows = {}
for n, r in (('SPEC3c (40 params)', s3), ('SPEC4 (29 params)', s4)):
    d = r['fold_aucs'] - cb15['fold_aucs']; e = r['fold_aucs'] - s3['fold_aucs']
    rows[n] = {'AUC (15)': r['auc_mean'], 'top-decile': r['top_decile_prec'], 'Δ vs CatBoost, seeds 1-2': d[5:].mean(), 'se': d[5:].std() / np.sqrt(10),
               'Δ vs SPEC3c, seeds 1-2': e[5:].mean(), 'se ': e[5:].std() / np.sqrt(10)}
pd.DataFrame(rows).T.style.format({'AUC (15)': '{:.4f}', 'top-decile': '{:.1%}', 'Δ vs CatBoost, seeds 1-2': '{:+.4f}', 'se': '{:.4f}',
                                   'Δ vs SPEC3c, seeds 1-2': '{:+.4f}', 'se ': '{:.4f}'})

,seed-0 AUC,seed-0 log-loss
0.010000,0.71954,0.31698
0.030000,0.72048,0.31664
0.100000,0.72063,0.31660
0.300000,0.72068,0.31660
1.000000,0.72068,0.31660
10.000000,0.72069,0.31660


,AUC (15),top-decile,"Δ vs CatBoost, seeds 1-2",se,"Δ vs SPEC3c, seeds 1-2",se
SPEC3c (40 params),0.7195,31.2%,+0.0052,0.0019,+0.0000,0.0000
SPEC4 (29 params),0.7206,31.1%,+0.0062,0.0018,+0.0010,0.0003


**Reading:** **SPEC4 has 29 parameters** (down from 40):
- one intercept for 2015–16;
- no missing-value flags;
- drivers-per-vehicle bands shared by 2015 and 2016;
- luxury share in 2017-18 only;
- package as one ordinal term;
- credit only as the 2015 step at 650.

It scores **15-fold AUC 0.7206**, **+0.0010 ± 0.0003 over SPEC3c on seeds 1–2** and +0.0062 over CatBoost. It stays insensitive to regularization (flat for C ≥ 0.1).

## E. Anything left? Deep residual models and targeted 3-way terms
Residual GBMs get the `SPEC4` log-odds as a fixed baseline, so they can only add what the formula misses. If 3+-way interactions existed, deeper trees should find more than shallow ones.

In [7]:
import catboost as cbm, lightgbm as lgb
LC = FE.add_cats(L); FX = PC['features']; cats = [c for c in FX if not pd.api.types.is_numeric_dtype(LC[c])]
folds = C.folds(lab)
BL = [(lambda m: (m.decision(L.iloc[tr]), m.decision(L.iloc[va])))(FM.Formula(SPEC4, 0.1, '2era').fit(L.iloc[tr], y[tr])) for s, k, tr, va in folds]
def resid(kind, depth):
    curves, fa = [], []
    for (s, k, tr, va), (btr, bva) in zip(folds, BL):
        if kind == 'CatBoost':
            r = cbm.CatBoostClassifier(**(PC['params'] | {'learning_rate': 0.02, 'depth': depth}), iterations=3000, early_stopping_rounds=200, use_best_model=False)
            r.fit(cbm.Pool(C.cb_X(LC.iloc[tr][FX], cats), y[tr], cat_features=cats, baseline=btr),
                  eval_set=cbm.Pool(C.cb_X(LC.iloc[va][FX], cats), y[va], cat_features=cats, baseline=bva))
            curves.append(np.array(r.get_evals_result()['validation']['AUC']))
        else:
            ev = {}
            lgb.train(dict(objective='binary', metric='auc', learning_rate=0.02, max_depth=depth, num_leaves=2 ** depth - 1, min_child_samples=50,
                           feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10, verbosity=-1, seed=0),
                      lgb.Dataset(L.iloc[tr][FS.F], y[tr], init_score=btr), 3000, valid_sets=[lgb.Dataset(L.iloc[va][FS.F], y[va], init_score=bva)],
                      callbacks=[lgb.early_stopping(200, verbose=False), lgb.record_evaluation(ev)])
            curves.append(np.array(ev['valid_0']['auc']))
        fa.append(roc_auc_score(y[va], bva))
    Lm = max(map(len, curves)); bi = int(np.mean([np.pad(c, (0, Lm - len(c)), mode='edge') for c in curves], axis=0).argmax())
    d = np.array([c[min(bi, len(c) - 1)] for c in curves]) - np.array(fa)
    return {'trees': bi + 1, 'Δ AUC (15)': d.mean(), 'se': d.std() / np.sqrt(15), 'Δ seeds 1-2': d[5:].mean(), 'se (1-2)': d[5:].std() / np.sqrt(10)}
re_ = pd.DataFrame({f'{k} depth {dpt}': resid(k, dpt) for k, dpt in (('CatBoost', 4), ('CatBoost', 6), ('CatBoost', 8), ('LightGBM', 6))}).T
display(re_.style.format({'trees': '{:.0f}', 'Δ AUC (15)': '{:+.5f}', 'se': '{:.5f}', 'Δ seeds 1-2': '{:+.5f}', 'se (1-2)': '{:.5f}'}))

b4 = run4(SPEC4); med = L.QUOTED_AMT.median()
X3 = {'age band × drivers/vehicle (0.5,1] (early/late)': (L.DRV_AGE_35_50 * L.R_DPV_HALF_TO_1, 'split'),
      'age band × more drivers than vehicles (early/late)': (L.DRV_AGE_35_50 * L.R_DPV_GT1, 'split'),
      'education majority × density (2017-18)': (L.R_EDU_MAJ * L.ACS_POP_DENSITY.fillna(0) / 1e4, 'late'),
      'education majority × drivers/vehicle (2017-18)': (L.R_EDU_MAJ * L.R_DPV_HALF_TO_1, 'late'),
      'education majority × age band (2017-18)': (L.R_EDU_MAJ * L.DRV_AGE_35_50, 'late'),
      'price × credit ≥ 650 (2015-16)': (L.QUOTED_AMT.fillna(med) * (L.CREDIT_SCORE >= 650) / 1e4, 'early'),
      'age band × number of drivers (early/late)': (L.DRV_AGE_35_50 * L.NUM_DRIVERS, 'split'),
      'discount × package (2016)': (L.DISCOUNT_IND * L.PKG_ORDINAL.fillna(0), '2016'),
      'package × CAT zone': (L.PKG_ORDINAL.fillna(0) * L.CAT_ZONE.fillna(3), 'all'),
      'price × age band (early/late)': (L.QUOTED_AMT.fillna(med) * L.DRV_AGE_35_50 / 1e4, 'split'),
      'density × age band (early/late)': (L.ACS_POP_DENSITY.fillna(0) * L.DRV_AGE_35_50 / 1e4, 'split'),
      'no prior carrier × discount (2016)': (L.PRIOR_CARRIER_GRP.astype(str).eq('Missing') * L.DISCOUNT_IND, '2016')}
rows = {}
for i, (n, (v, sc)) in enumerate(X3.items()):
    col = f'X3_{i}'; L[col] = v.astype(float)
    rows[n] = cmp(run4(SPEC4 + [t(col, 'linear', sc)]), b4)
tab(rows).style.format(fmt)

,trees,Δ AUC (15),se,Δ seeds 1-2,se (1-2)
CatBoost depth 4,71,+0.00025,0.00015,+0.00025,0.00014
CatBoost depth 6,23,+0.00018,0.00010,+0.00021,0.00009
CatBoost depth 8,17,+0.00022,0.00009,+0.00019,0.00013
LightGBM depth 6,6,+0.00012,0.00006,+0.00007,0.00007


,auc,Δauc,se_auc,Δll(+better),se_ll,z_ll
"age band × drivers/vehicle (0.5,1] (early/late)",0.72083,+0.00020,0.00026,+0.25,0.61,+0.4
age band × more drivers than vehicles (early/late),0.72052,-0.00010,0.00020,-0.34,0.53,-0.7
education majority × density (2017-18),0.72055,-0.00007,0.00008,-0.17,0.42,-0.4
education majority × drivers/vehicle (2017-18),0.72063,+0.00001,0.00002,-0.16,0.07,-2.4
education majority × age band (2017-18),0.72057,-0.00005,0.00005,-0.23,0.11,-2.1
price × credit ≥ 650 (2015-16),0.72050,-0.00012,0.00006,-0.40,0.17,-2.4
age band × number of drivers (early/late),0.72068,+0.00005,0.00013,-0.30,0.55,-0.6
discount × package (2016),0.72064,+0.00001,0.00018,-0.19,0.80,-0.2
package × CAT zone,0.72067,+0.00005,0.00005,-0.06,0.14,-0.4
price × age band (early/late),0.72060,-0.00003,0.00023,-0.73,0.84,-0.9


**Reading:**
- **No 3+-way structure:** residual CatBoost gains the same tiny +0.0002 at depth 4, 6 and 8. If higher-order interactions existed, deeper trees would find more. LightGBM depth 6 adds +0.0001.
- **No targeted 3-way term helps:** all 12 are within ±0.0002, and several are clearly worse.
- **What's left** (~0.0002, ~2 se) is too diffuse to name; a blend with CatBoost captures it.

## F. Blend and submissions
Components on the same 15 folds: CatBoost `07` and the formula `SPEC4`. Weight searched on seed 0, judged on seeds 1–2 against the current submission `blend_v3` (0.7 × `SPEC3c` + 0.3 × CatBoost).

In [8]:
FOLDS = C.folds(lab)
aucs = lambda P: np.array([roc_auc_score(y[va], P[s, va]) for s, k, tr, va in FOLDS])
v3 = 0.7 * s3['oof_all'] + 0.3 * cb15['oof_all']
W = np.round(np.arange(0, 1.0001, 0.05), 2)
s0 = [aucs(w * s4['oof_all'] + (1 - w) * cb15['oof_all'])[:5].mean() for w in W]
wf = float(W[int(np.argmax(s0))])
cands = {'blend_v3 (current)': v3, 'SPEC4 alone': s4['oof_all'], f'{wf:.2f} SPEC4 + {1 - wf:.2f} CatBoost': wf * s4['oof_all'] + (1 - wf) * cb15['oof_all']}
base = aucs(v3); rows = {}
for n, P in cands.items():
    a = aucs(P); d = a - base
    rows[n] = {'AUC (15)': a.mean(), 'Δ vs blend_v3': d.mean(), 'se': d.std() / np.sqrt(15), 'Δ seeds 1-2': d[5:].mean(), 'se (1-2)': d[5:].std() / np.sqrt(10),
               'top-decile': np.mean([C.top_decile_precision(y, P[s]) for s in range(3)])}
tf = pd.DataFrame(rows).T
display(tf.style.format({'AUC (15)': '{:.4f}', 'Δ vs blend_v3': '{:+.4f}', 'se': '{:.4f}', 'Δ seeds 1-2': '{:+.4f}', 'se (1-2)': '{:.4f}', 'top-decile': '{:.1%}'}))
best_n = tf.index[-1]
adopt = tf.loc[best_n, 'Δ seeds 1-2'] > 2 * tf.loc[best_n, 'se (1-2)']
print('adopt blend_v4:', adopt)

,AUC (15),Δ vs blend_v3,se,Δ seeds 1-2,se (1-2),top-decile
blend_v3 (current),0.7204,+0.0000,0.0000,+0.0000,0.0000,31.0%
SPEC4 alone,0.7206,+0.0002,0.0004,+0.0001,0.0005,31.1%
0.75 SPEC4 + 0.25 CatBoost,0.7212,+0.0008,0.0002,+0.0008,0.0002,31.1%


adopt blend_v4: True


In [9]:
H = FM.prep(hold)
final = FM.Formula(SPEC4, 0.1, '2era').fit(L, y)
pf = final.predict(H)
out = {'formula_v2': C.write_submission('formula_v2', hold, pf)}
if adopt:
    cbp = pd.read_csv(sorted(glob.glob('../submissions/catboost_cats5_*.csv'))[-1]); assert (cbp.POLICY_ID.values == hold.POLICY_ID.values).all()
    p = wf * pf + (1 - wf) * cbp.CONVERT_IND_PRED.to_numpy()
    out['blend_v4'] = C.write_submission('blend_v4', hold, p)
    print(f'blend_v4: {pd.Series(p).nunique()} distinct, mean {p.mean():.4f}')
print({k: v.name for k, v in out.items()}, f'| formula_v2: {pd.Series(pf).nunique()} distinct, mean {pf.mean():.4f}')
coef = final.coef_table()
json.dump({'spec': SPEC4, 'C': 0.1, 'intercepts': '2era', 'n_params': final.n_params, 'cv_auc_15': s4['auc_mean'],
           'cv_top_decile': s4['top_decile_prec'], 'coefficients': coef['coef'].round(6).to_dict(),
           'blend_weight_formula': wf if adopt else None, 'submissions': {k: v.name for k, v in out.items()}},
          open('formula_v2_spec.json', 'w'), indent=1)
coef[['coef']].style.format('{:+.6f}')

blend_v4: 7375 distinct, mean 0.1116
{'formula_v2': 'formula_v2_20261031.csv', 'blend_v4': 'blend_v4_20261031.csv'} | formula_v2: 7375 distinct, mean 0.1117


,coef
term,
era=2017-18,-1.620451
CREDIT_SCORE:step[650]>=650 @2015,+0.332867
QUOTED_AMT:linear @2015,-0.000044
QUOTED_AMT:linear @2016,-0.000078
QUOTED_AMT:linear @2017-18,-0.000016
ACS_POP_DENSITY:linear @2015,+0.000008
ACS_POP_DENSITY:linear @2016,+0.000011
ACS_POP_DENSITY:linear @2017-18,+0.000036
R_EDU_MAJ:flag @2017-18,+1.189820


## Conclusions
- **Final formula `SPEC4`: 29 parameters, 15-fold AUC 0.7206**, top-decile precision 31.1%. +0.0062 over CatBoost and +0.0010 over `SPEC3c` on the unseen seeds 1–2.
- **The generator, as recovered:** the log-odds of converting are
  - **intercept:** −1.62 from 2017;
  - **+0.67** if the mean driver age is in [35, 50];
  - **drivers per vehicle:** +0.34 / +0.74 (2015-16 / 2017-18) if in (0.5, 1]; −0.29 / −0.48 if more drivers than vehicles;
  - **CAT zone:** −0.13 per zone;
  - **package:** −0.13 per level (Low → High);
  - **prior carrier:** −0.26 to +0.60;
  - **number of drivers:** −0.20 per driver in 2015-16, +0.08 in 2017-18;
  - **price:** −0.044 / −0.078 / −0.016 per $1,000 (2015 / 2016 / 2017-18);
  - **density:** +0.008 / +0.011 / +0.036 per 1,000 people per sq mi;
  - **+0.33** if credit ≥ 650 (2015 only);
  - **+0.71** with a discount (2016 only);
  - **+1.19** if more than half the drivers are educated (2017-18 only);
  - **+0.33** × luxury share (2017-18 only).
- **Nothing more found:**
  - no hidden columns;
  - no year split within 2017-18;
  - no better cut points;
  - no 3+-way interactions.
- **Submissions:**
  - **`blend_v4_20261031.csv` = 0.75 × SPEC4 + 0.25 × CatBoost (07):** 15-fold AUC **0.7212**, +0.0008 ± 0.0002 over `blend_v3` on seeds 1–2. **Current best.**
  - **`formula_v2_20261031.csv`:** the formula alone (0.7206).
- Saved: `formula_v2_spec.json` (spec, coefficients) and `formula_steps.SPEC4`.